# 자동 fleet 인덱싱 — CPU / Colab
프로젝트 source와 의존성을 준비한 뒤 사용합니다. 기본 status는 원본 요청을 하지 않습니다. 명시 metadata pin·검토된 estimates·bounded_auto_v1 full plan·정확한 plan SHA가 필요하며 미측정 전체 계획은 실행할 수 없습니다.
auto-run 한 번으로 예산 내 구간·archive를 자동 저장/정산합니다. stop은 최대 한 bounded segment 뒤 반영되는 협력 요청이며 강제 종료의 pending 소비는 자동 복구하지 않습니다. resume은 같은 store/plan에서 stop을 확인하고 남은 예산으로 진행합니다.
1MiB chunk reader를 구간마다 재시작하는 bounded 경로입니다. 진짜 stateful 해제 엔진이나 전체 CRC·샘플 pairing 완료를 뜻하지 않습니다.
auto-run/resume은 별도 CPU subprocess를 시작하고 셀은 즉시 반환합니다. 같은 notebook에서 ACTION=status/stop으로 마지막 셀을 다시 실행할 수 있습니다. worker는 PROJECT_DIR을 cwd로 쓰며 부모의 환경·ADC 파일을 공유합니다. 먼저 Colab 인증과 source/의존성을 준비해야 합니다. 런타임 종료 시 worker도 중단되며 숨은 자동 재시작은 없습니다.
WORKER_LOG에는 stdout/stderr, WORKER_REPORT에는 정상/실패 보고서가 남습니다. status는 종료코드와 경로도 표시합니다. 강제 종료로 보고서가 없으면 coverage의 HEAD/pending/lease를 보존해 명시 복구 절차로 넘깁니다. mounted store에서 정상 commit을 확인하며 전체 store를 회수 ZIP으로 복제하지 않습니다.


In [ ]:
from pathlib import Path
import sys, uuid
PROJECT_DIR = Path("/content/rewind_source")
INVENTORY = Path("/content/runtime_inventory.json")
PLAN = Path("/content/reviewed_auto_plan.json")
EXPECTED_PLAN_SHA256 = ""  # 검토된 고정 plan hash
STORE = Path("/content/drive/MyDrive/fleet_auto_store")
WORKSPACE = Path("/content/fleet_auto_workspace")
ACTION = "status"  # status / coverage / auto-run / stop / resume
SESSION_SECONDS = 900
MAX_SEGMENTS = 1000000  # 예산/시간/진전 없음이 먼저 실행을 제한합니다.
OWNER = "colab_auto"


In [ ]:
import subprocess, os
if str(PROJECT_DIR) not in sys.path: sys.path.insert(0, str(PROJECT_DIR))
from rewind_fleet.__main__ import main
if ACTION not in ("status", "coverage", "auto-run", "stop", "resume"): raise ValueError("invalid_action")
args = [ACTION, "--store", str(STORE)]
if ACTION in ("auto-run", "stop", "resume"):
    if len(EXPECTED_PLAN_SHA256) != 64: raise ValueError("reviewed_plan_pin_required")
    args += ["--plan-sha256", EXPECTED_PLAN_SHA256]
if ACTION in ("auto-run", "resume"):
    if globals().get("WORKER") is not None and WORKER.poll() is None: raise ValueError("worker_already_running")
    WORKSPACE.mkdir(parents=True, exist_ok=True)
    job = uuid.uuid4().hex
    WORKER_REPORT = WORKSPACE / ("run-" + job + ".json")
    WORKER_LOG = WORKSPACE / ("worker-" + job + ".log")
    args += ["--inventory", str(INVENTORY), "--plan", str(PLAN), "--workspace", str(WORKSPACE), "--owner", OWNER, "--single-writer", "--max-segments", str(MAX_SEGMENTS), "--session-seconds", str(SESSION_SECONDS), "--output", str(WORKER_REPORT)]
    with WORKER_LOG.open("xb") as log:
        WORKER = subprocess.Popen([sys.executable, "-u", "-m", "rewind_fleet", *args], cwd=PROJECT_DIR, stdout=log, stderr=subprocess.STDOUT, creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0) if os.name == "nt" else 0)
    print({"pid": WORKER.pid, "log": str(WORKER_LOG), "report": str(WORKER_REPORT), "status_action": "status"})
else:
    main(args)
    if globals().get("WORKER") is not None:
        print({"worker_exit_code": WORKER.poll(), "log": str(WORKER_LOG), "report": str(WORKER_REPORT), "report_exists": WORKER_REPORT.is_file()})
